In [ ]:
import os
os.environ['CUDA_DEVICE_ORDER'] = 'PCI_BUS_ID'
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

In [ ]:
import random
import torch
import itertools
from tqdm import tqdm
import functools
from PIL import Image
from collections import defaultdict
import base64
import io
from torch.utils.data import DataLoader
import torch.nn.functional as F
from transformers import get_scheduler
from torch import optim
import torch.nn as nn
import numpy as np
from torch.optim.lr_scheduler import OneCycleLR
import matplotlib.pyplot as plt
import psycopg
from pathlib import Path
import gzip
import pickle
import math
from dataclasses import dataclass

In [ ]:
#trueskill-checkpoints/trueskill_0100000_20250803_211614.pkl.gz


CKPT_DIR = Path("trueskill-checkpoints")

@dataclass
class SkillRating:
	mu: float
	sigma: float


def load_state_pickle(path: Path):
	with gzip.open(path, "rb") as fp:
		payload = pickle.load(fp)
	ratings = {k: SkillRating(mu=v[0], sigma=v[1]) for k, v in payload["ratings"].items()}
	pair_counts = payload["pair_counts"]
	mu_deltas = payload["mu_deltas"]
	step = payload["step"]
	beta = payload['beta']
	return ratings, pair_counts, mu_deltas, step, beta

ratings, pair_counts, mu_deltas, step, openskill_model_beta = load_state_pickle(CKPT_DIR / "trueskill_0100000_20250803_230740.pkl.gz")

In [ ]:
with psycopg.connect(dbname='postgres', user='postgres', host=str(Path.cwd().parent / "pg-socket")) as conn:
	cursor = conn.cursor()
	filehash_to_path = {}
	embeddings = {}
	all_filehashes = set(ratings.keys())

	for filehash in tqdm(all_filehashes, desc="Loading paths"):
		cursor.execute('SELECT path, embedding FROM images WHERE filehash = %s', (filehash,))
		path, embedding = cursor.fetchone()
		filehash_to_path[filehash] = path
		embedding = bytes(embedding)
		embedding = torch.frombuffer(embedding, dtype=torch.float16).to(torch.float32)
		embeddings[filehash] = embedding

In [ ]:
class RankingDataset(torch.utils.data.Dataset):
	def __init__(self, data):
		self.data = data

	def __len__(self):
		return len(self.data)

	def __getitem__(self, idx):
		return self.data[idx]


ELO_SCALE = 400 / math.log(10)
ELO_OFFSET = 1500
def mu_to_elo(mu: float):
	return mu * (ELO_SCALE / openskill_model_beta) + ELO_OFFSET

scores = {p: mu_to_elo(rtg.mu) for p, rtg in ratings.items()}
min_score = min(scores.values())
max_score = max(scores.values())
NUM_BINS = 10
bin_size = (max_score - min_score) / NUM_BINS

rankings = {p: min(NUM_BINS - 1, int((r - min_score) / bin_size)) for p, r in scores.items()}

data = [(embeddings[p], rank) for p, rank in rankings.items()]
random.shuffle(data)
test_size = len(data) // 10
train_data = data[:-test_size]
test_data = data[-test_size:]

train_dataset = RankingDataset(train_data)
test_dataset = RankingDataset(test_data)

print(len(train_dataset), len(test_dataset))

In [ ]:
class ScoreClassifier(nn.Module):
	def __init__(self, embedding_size: int, dropout: float, bins: int):
		super(ScoreClassifier, self).__init__()

		self.ln = nn.LayerNorm(embedding_size)
		self.dropout1 = nn.Dropout(dropout * 0.5)
		self.linear1 = nn.Linear(embedding_size, embedding_size*2)
		#self.act_fn = nn.GELU()
		self.act_fn = nn.SiLU()
		self.dropout2 = nn.Dropout(dropout)
		self.linear2 = nn.Linear(embedding_size*2, bins)
	
	def forward(self, x: torch.Tensor):
		x = self.ln(x)
		x = self.dropout1(x)
		x = self.linear1(x)

		x = self.act_fn(x)
		x = self.dropout2(x)
		x = self.linear2(x)

		return x

## LR Finder

In [ ]:
def lr_finder(
	batch_size: int,
	dropout: float,
	weight_decay: float,
	start_lr: float = 1e-7,
	end_lr: float = 10,
	num_it: int = 100,
	beta: float = 0.98,
):
	learning_rates = np.geomspace(start_lr, end_lr, num_it)
	model = ScoreClassifier(768, dropout=dropout, bins=NUM_BINS).to('cuda')

	loss_function = nn.CrossEntropyLoss()
	optimizer = optim.AdamW(model.parameters(), lr=1.0, weight_decay=weight_decay)

	train_dataloader = DataLoader(
		train_dataset,
		batch_size=batch_size,
		shuffle=True,
		drop_last=True,
	)

	dataset_iter = iter(train_dataloader)
	training_losses = []
	training_lrs = []
	smooth_loss = None
	avg_loss = None
	best_loss = 1000000000

	for lr in learning_rates:
		# Set learning rate
		for param_group in optimizer.param_groups:
			param_group['lr'] = lr

		# Get batch
		try:
			x, y = next(dataset_iter)
		except StopIteration:
			dataset_iter = iter(train_dataloader)
			x, y = next(dataset_iter)
		
		x = x.to('cuda')
		y = y.to('cuda')
		
		# Train
		model.train()
		output = model(x)
		loss = loss_function(output, y)
		optimizer.zero_grad(set_to_none=True)
		loss.backward()
		optimizer.step()

		training_losses.append(loss.item())
		training_lrs.append(lr)

		if avg_loss is None:
			avg_loss = loss
		else:
			avg_loss = beta * avg_loss + (1 - beta) * loss
		
		smooth_loss = avg_loss / (1 - beta ** (len(training_losses) + 1))
		
		best_loss = min(best_loss, smooth_loss)

		if smooth_loss > 4 * best_loss or torch.isnan(smooth_loss):
			break
	
	return {
		"training_losses": training_losses,
		"training_lrs": training_lrs,
	}


results = lr_finder(
	batch_size=512,
	dropout=0.1,
	weight_decay=2.0,
)

plt.plot(results['training_lrs'][:-2], results['training_losses'][:-2])
plt.xscale('log')
plt.yscale('log')
plt.show()

In [ ]:
@torch.no_grad()
def test(model, dataloader: DataLoader, loss_function) -> tuple[float, float]:
	model.eval()

	corrects = []
	loss_sum = torch.zeros(1, dtype=torch.float32, device='cpu', requires_grad=False)

	for x, y in dataloader:
		x = x.to('cuda')
		y = y.to('cuda')
		output = model(x)
		loss = loss_function(output, y)
		loss_sum.add_(loss.detach().cpu())
		probs = F.softmax(output, dim=1)
		predictions = torch.argmax(probs, dim=1)
		correct = (predictions == y).tolist()
		corrects.extend(correct)
	
	mean_loss = loss_sum / len(dataloader)
	return sum(corrects) / len(corrects), mean_loss.item()


def train_a_model(
	batch_size: int,
	learning_rate: float,
	dropout: float,
	warmup_samples: int,
	training_samples: int,
	test_every: int,
	weight_decay: float,
	one_cycle: bool,
	prodigy: bool = False,
	label_smoothing: float = 0.0,
):
	model = ScoreClassifier(768, dropout=dropout, bins=NUM_BINS).to('cuda')

	train_dataloader = DataLoader(
		dataset=train_dataset,
		batch_size=batch_size,
		shuffle=True,
	)

	test_dataloader = DataLoader(
		dataset=test_dataset,
		batch_size=batch_size,
		shuffle=False,
		drop_last=False,
	)

	warmup_steps = warmup_samples // batch_size
	training_steps = max(1, training_samples // batch_size)
	test_every_steps = max(1, test_every // batch_size)

	loss_function = nn.CrossEntropyLoss()
	if prodigy:
		import prodigyopt
		optimizer = prodigyopt.Prodigy(params=model.parameters(), lr=learning_rate, weight_decay=weight_decay)
		lr_scheduler = get_scheduler(
			name="cosine",
			optimizer=optimizer,
			num_warmup_steps=0,
			num_training_steps=training_steps,
		)
	else:
		optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
		if one_cycle:
			lr_scheduler = OneCycleLR(
				optimizer=optimizer,
				max_lr=learning_rate,
				total_steps=training_steps,
			)
		else:
			lr_scheduler = get_scheduler(
				name="cosine",
				optimizer=optimizer,
				num_warmup_steps=warmup_steps,
				num_training_steps=training_steps,
			)

	dataset_iter = iter(train_dataloader)
	training_losses = []
	training_lrs = []
	test_losses = []
	test_accuracies = []

	for step in range(training_steps):
		# Get batch
		try:
			x, y = next(dataset_iter)
		except StopIteration:
			dataset_iter = iter(train_dataloader)
			x, y = next(dataset_iter)
		
		x = x.to('cuda')
		y = y.to('cuda')

		if label_smoothing > 0.0:
			# Apply label smoothing
			y_one_hot = F.one_hot(y, num_classes=NUM_BINS).float()
			y_smooth = (1 - label_smoothing) * y_one_hot + (label_smoothing / NUM_BINS)
			y = y_smooth
		
		# Train
		model.train()
		output = model(x)
		loss = loss_function(output, y)
		optimizer.zero_grad(set_to_none=True)
		loss.backward()
		optimizer.step()
		lr_scheduler.step()

		# Test
		if step % test_every_steps == 0:
			test_acc, test_loss = test(model, test_dataloader, loss_function)
			test_losses.append(test_loss)
			test_accuracies.append(test_acc)
		
		training_losses.append(loss.item())
		training_lrs.append(lr_scheduler.get_last_lr()[0])
	
	return {
		"training_losses": training_losses,
		"training_lrs": training_lrs,
		"test_losses": test_losses,
		"test_accuracies": test_accuracies,
		"model": model,
	}

In [ ]:
result = train_a_model(
	batch_size=512,
	learning_rate=8e-3,
	dropout=0.1,
	warmup_samples=1000,
	training_samples=80000,
	test_every=512,
	weight_decay=2.0,
	one_cycle=True,
	prodigy=False,
	label_smoothing=0.0,
)

min_test_loss = result["test_losses"][-1]
min_test_accuracy = result["test_accuracies"][-1]


plt.plot(result['training_losses'])
plt.title('Training loss')
plt.show()

plt.plot(result['training_lrs'])
plt.title('Learning rate')
plt.show()

plt.plot(result['test_losses'])
plt.title('Test loss')
plt.show()

plt.plot(result['test_accuracies'])
plt.title('Test accuracy')
plt.show()

print(f"Test loss: {min_test_loss}")
print(f"Test accuracy: {min_test_accuracy}")

best_model = result

## Save

In [ ]:
torch.save(best_model['model'].state_dict(), "scorer.pt")